In [1]:
import os
from pathlib import Path

import nibabel as nib
import torch
from nibabel.filebasedimages import ImageFileError
from totalsegmentator.config import has_valid_license_offline, set_license_number
from totalsegmentator.python_api import totalsegmentator

NOTEBOOK_DIR = Path.cwd()
if NOTEBOOK_DIR.name != "notebooks":
    NOTEBOOK_DIR = NOTEBOOK_DIR / "notebooks"

dataset_dir = NOTEBOOK_DIR / "nifti_data"
TASK = "heartchambers_highres"
START_FROM_TAVI = "TAVI_318"
LICENSE_NUMBER = "aca_BMG3E3I7YN8V5Z"

if not dataset_dir.exists():
    raise FileNotFoundError(f"Dataset directory not found: {dataset_dir}")

if LICENSE_NUMBER:
    set_license_number(LICENSE_NUMBER, skip_validation=True)

license_status, license_message = has_valid_license_offline()
if license_status != "yes":
    raise RuntimeError(
        "The TotalSegmentator task 'heartchambers_highres' requires a TotalSegmentator license "
        "to create heart_myocardium.nii.gz. Get the free academic license from "
        "https://backend.totalsegmentator.com/license-academic/ and either set the "
        "TOTALSEG_LICENSE_NUMBER environment variable before starting Jupyter or run in a notebook cell: "
        "import os; os.environ['TOTALSEG_LICENSE_NUMBER'] = '<your-license-number>'. "
        f"Current license status: {license_status}; {license_message}"
    )

if not torch.backends.mps.is_available():
    raise RuntimeError(
        "MPS is not available in this Python environment. "
        "Run this notebook on Apple Silicon with an MPS-enabled PyTorch build."
    )

patient_dirs = sorted(p for p in dataset_dir.iterdir() if p.is_dir() and p.name.startswith("TAVI_"))
if START_FROM_TAVI:
    patient_dirs = [p for p in patient_dirs if p.name >= START_FROM_TAVI]
    print(f"Starting from {START_FROM_TAVI}: {len(patient_dirs)} patient folders queued")
processed = []
skipped = []
missing_ct = []
unreadable_ct = []

for patient_dir in patient_dirs:
    ct_file = patient_dir / "CT_LATE.nii.gz"
    output_dir = patient_dir / "TotalSegmentator" / "CT_LATE" / TASK
    myocardium_mask = output_dir / "heart_myocardium.nii.gz"

    if not ct_file.exists():
        missing_ct.append(patient_dir.name)
        continue

    try:
        nib.load(str(ct_file))
    except ImageFileError as exc:
        unreadable_ct.append((patient_dir.name, str(exc)))
        print(f"Skipping {patient_dir.name}: CT_LATE unreadable ({exc})")
        continue

    if myocardium_mask.exists():
        skipped.append(patient_dir.name)
        continue

    output_dir.mkdir(parents=True, exist_ok=True)
    print(f"Processing {patient_dir.name} -> {myocardium_mask}")

    try:
        totalsegmentator(
            input=str(ct_file),
            output=str(output_dir),
            task="heartchambers_highres",
            device="mps",
            quiet=False,
        )
    except SystemExit as exc:
        raise RuntimeError(
            f"TotalSegmentator failed for {patient_dir.name}. "
            "Check that the heartchambers_highres license is configured and valid, "
            "then rerun this cell."
        ) from exc
    processed.append(patient_dir.name)

print(f"Processed: {len(processed)}")
print(f"Skipped existing masks: {len(skipped)}")
print(f"Missing CT_LATE files: {len(missing_ct)}")
print(f"Unreadable CT_LATE files: {len(unreadable_ct)}")
for patient_id, reason in unreadable_ct:
    print(f"  {patient_id}: {reason}")

Starting from TAVI_318: 34 patient folders queued
Processing TAVI_318 -> /Users/ricca/Desktop/Health_Informatics_Internship_2026/notebooks/nifti_data/TAVI_318/TotalSegmentator/CT_LATE/heartchambers_highres/heart_myocardium.nii.gz

If you use this tool please cite: https://pubs.rsna.org/doi/10.1148/ryai.230024

Generating rough segmentation for cropping...
  (Using more robust (but slower) 3mm model for cropping.)
Resampling...
  Resampled in 0.55s
Predicting...


100%|██████████| 1/1 [00:01<00:00,  1.66s/it]


  Predicted in 5.56s
Resampling...
  cropping from (512, 512, 57) to (512, 442, 54)
Predicting...


100%|██████████| 18/18 [00:15<00:00,  1.16it/s]


  Predicted in 24.74s
Applying postprocessing: remove outside of crop mask...
  Applied in 0.11s
Saving segmentations...
Creating heart_atrium_left.nii.gzCreating heart_ventricle_right.nii.gzCreating heart_atrium_right.nii.gz
Creating heart_myocardium.nii.gz


Creating heart_ventricle_left.nii.gz
Creating aorta.nii.gz
Creating pulmonary_artery.nii.gz
  Saved in 3.62s
Processing TAVI_320 -> /Users/ricca/Desktop/Health_Informatics_Internship_2026/notebooks/nifti_data/TAVI_320/TotalSegmentator/CT_LATE/heartchambers_highres/heart_myocardium.nii.gz

If you use this tool please cite: https://pubs.rsna.org/doi/10.1148/ryai.230024

Generating rough segmentation for cropping...
  (Using more robust (but slower) 3mm model for cropping.)
Resampling...
  Resampled in 0.57s
Predicting...


...

  Predicted in 19.07s
Applying postprocessing: remove outside of crop mask...
  Applied in 0.11s
Saving segmentations...
Creating heart_ventricle_right.nii.gz
Creating heart_myocardium.nii.gzCreating heart_atrium_left.nii.gz

Creating aorta.nii.gz
Creating heart_ventricle_left.nii.gz
Creating heart_atrium_right.nii.gz
Creating pulmonary_artery.nii.gz
  Saved in 3.25s
Processing TAVI_332 -> /Users/ricca/Desktop/Health_Informatics_Internship_2026/notebooks/nifti_data/TAVI_332/TotalSegmentator/CT_LATE/heartchambers_highres/heart_myocardium.nii.gz

If you use this tool please cite: https://pubs.rsna.org/doi/10.1148/ryai.230024

Generating rough segmentation for cropping...
  (Using more robust (but slower) 3mm model for cropping.)
Resampling...
  Resampled in 0.47s
Predicting...


100%|██████████| 1/1 [00:00<00:00,  1.44it/s]


  Predicted in 4.13s
Resampling...
  cropping from (512, 512, 46) to (512, 502, 46)
Predicting...


100%|██████████| 12/12 [00:10<00:00,  1.19it/s]


  Predicted in 18.94s
Applying postprocessing: remove outside of crop mask...
  Applied in 0.10s
Saving segmentations...
Creating heart_myocardium.nii.gz
Creating heart_atrium_left.nii.gz
Creating aorta.nii.gz
Creating heart_atrium_right.nii.gz
Creating heart_ventricle_left.nii.gz
Creating heart_ventricle_right.nii.gz
Creating pulmonary_artery.nii.gz
  Saved in 3.27s
Processed: 3
Skipped existing masks: 31
Missing CT_LATE files: 0
Unreadable CT_LATE files: 0


## Compare TotalSegmentator myocardium masks to ground truth

This section evaluates `heart_myocardium.nii.gz` against each patient's `registration_mask.nii.gz`. Dice, IoU, and volume use the full binary masks. The table includes both `hd95_surface_mm`, computed after hollowing masks with binary erosion, and `hd95_full_mask_mm`, computed from every foreground voxel for comparison.


In [2]:
from pathlib import Path

import numpy as np
import pandas as pd
import nibabel as nib
from nibabel.filebasedimages import ImageFileError
from scipy import ndimage as ndi
from scipy.spatial import cKDTree

NOTEBOOK_DIR = Path.cwd()
if NOTEBOOK_DIR.name != "notebooks":
    NOTEBOOK_DIR = NOTEBOOK_DIR / "notebooks"

DATASET_DIR = NOTEBOOK_DIR / "nifti_data"
TOTAL_SEGMENTATOR_MASK_RELATIVE_PATH = Path(
    "TotalSegmentator/CT_LATE/heartchambers_highres/heart_myocardium.nii.gz"
)
GROUND_TRUTH_MASK_NAME = "registration_mask.nii.gz"
EVALUATION_OUTPUT_CSV = NOTEBOOK_DIR / "total_segmentator_heart_myocardium_metrics.csv"
SKIPPED_OUTPUT_CSV = NOTEBOOK_DIR / "total_segmentator_heart_myocardium_skipped.csv"


def load_binary_mask(mask_path):
    """Load a NIfTI mask as a boolean array plus its affine and voxel spacing."""
    image = nib.load(str(mask_path))
    data = np.asanyarray(image.dataobj)
    mask = np.nan_to_num(data, nan=0.0) > 0
    spacing = np.array(image.header.get_zooms()[: mask.ndim], dtype=float)
    return mask, image.affine, spacing


def hollow_surface(mask):
    """Keep only boundary voxels by subtracting an eroded copy from the mask."""
    if not mask.any():
        return np.zeros_like(mask, dtype=bool)
    structure = ndi.generate_binary_structure(mask.ndim, 1)
    eroded = ndi.binary_erosion(mask, structure=structure, border_value=0)
    return mask & ~eroded


def symmetric_hd95_mm(pred_mask, gt_mask, spacing):
    """Compute symmetric 95th percentile nearest-neighbor distance in millimeters."""
    if not pred_mask.any() or not gt_mask.any():
        return np.nan

    pred_points_mm = np.argwhere(pred_mask) * spacing
    gt_points_mm = np.argwhere(gt_mask) * spacing

    pred_to_gt = cKDTree(gt_points_mm).query(pred_points_mm, k=1)[0]
    gt_to_pred = cKDTree(pred_points_mm).query(gt_points_mm, k=1)[0]
    return float(np.percentile(np.concatenate([pred_to_gt, gt_to_pred]), 95))


def surface_hd95_mm(pred_mask, gt_mask, spacing):
    """Compute symmetric HD95 in millimeters using only hollowed mask surface voxels."""
    return symmetric_hd95_mm(hollow_surface(pred_mask), hollow_surface(gt_mask), spacing)


def full_mask_hd95_mm(pred_mask, gt_mask, spacing):
    """Compute symmetric HD95 in millimeters using every foreground mask voxel."""
    return symmetric_hd95_mm(pred_mask, gt_mask, spacing)


def compare_masks(patient_dir):
    """Compare one TotalSegmentator mask with the corresponding ground truth mask."""
    patient_id = patient_dir.name
    pred_path = patient_dir / TOTAL_SEGMENTATOR_MASK_RELATIVE_PATH
    gt_path = patient_dir / GROUND_TRUTH_MASK_NAME

    if not pred_path.exists():
        return None, {"patient_id": patient_id, "reason": "missing_totalsegmentator_mask", "path": str(pred_path)}
    if not gt_path.exists():
        return None, {"patient_id": patient_id, "reason": "missing_ground_truth_mask", "path": str(gt_path)}

    try:
        pred_mask, pred_affine, pred_spacing = load_binary_mask(pred_path)
        gt_mask, gt_affine, gt_spacing = load_binary_mask(gt_path)
    except (ImageFileError, OSError, ValueError) as exc:
        return None, {"patient_id": patient_id, "reason": "unreadable_mask", "path": str(patient_dir), "error": str(exc)}

    if pred_mask.shape != gt_mask.shape:
        return None, {
            "patient_id": patient_id,
            "reason": "shape_mismatch",
            "pred_shape": pred_mask.shape,
            "gt_shape": gt_mask.shape,
        }

    if not np.allclose(pred_affine, gt_affine, rtol=1e-3, atol=1e-3):
        return None, {"patient_id": patient_id, "reason": "affine_mismatch"}

    intersection = int(np.logical_and(pred_mask, gt_mask).sum())
    pred_voxels = int(pred_mask.sum())
    gt_voxels = int(gt_mask.sum())
    union = int(np.logical_or(pred_mask, gt_mask).sum())
    pred_surface_voxels = int(hollow_surface(pred_mask).sum())
    gt_surface_voxels = int(hollow_surface(gt_mask).sum())

    dice = np.nan if (pred_voxels + gt_voxels) == 0 else (2.0 * intersection) / (pred_voxels + gt_voxels)
    iou = np.nan if union == 0 else intersection / union

    voxel_volume_ml = float(np.prod(gt_spacing) / 1000.0)
    pred_volume_ml = pred_voxels * voxel_volume_ml
    gt_volume_ml = gt_voxels * voxel_volume_ml

    row = {
        "patient_id": patient_id,
        "dice": dice,
        "iou": iou,
        "hd95_surface_mm": surface_hd95_mm(pred_mask, gt_mask, gt_spacing),
        "hd95_full_mask_mm": full_mask_hd95_mm(pred_mask, gt_mask, gt_spacing),
        "totalsegmentator_volume_ml": pred_volume_ml,
        "ground_truth_volume_ml": gt_volume_ml,
        "volume_difference_ml": pred_volume_ml - gt_volume_ml,
        "volume_ratio_totalsegmentator_to_gt": np.nan if gt_volume_ml == 0 else pred_volume_ml / gt_volume_ml,
        "totalsegmentator_voxels": pred_voxels,
        "ground_truth_voxels": gt_voxels,
        "totalsegmentator_surface_voxels": pred_surface_voxels,
        "ground_truth_surface_voxels": gt_surface_voxels,
        "intersection_voxels": intersection,
        "union_voxels": union,
        "voxel_spacing_mm": "x".join(f"{value:g}" for value in gt_spacing),
        "totalsegmentator_mask": str(pred_path),
        "ground_truth_mask": str(gt_path),
    }
    return row, None


In [3]:
# Surface-only HD95 sanity check
filled_cube = np.ones((3, 3, 3), dtype=bool)
surface_cube = hollow_surface(filled_cube)

surface_check_df = pd.DataFrame(
    [
        {
            "mask": "3x3x3 filled cube",
            "full_foreground_voxels": int(filled_cube.sum()),
            "surface_only_voxels": int(surface_cube.sum()),
            "removed_interior_voxels": int(filled_cube.sum() - surface_cube.sum()),
            "surface_hd95_self_mm": surface_hd95_mm(filled_cube, filled_cube, np.array([1.0, 1.0, 1.0])),
            "full_mask_hd95_self_mm": full_mask_hd95_mm(filled_cube, filled_cube, np.array([1.0, 1.0, 1.0])),
        }
    ]
)
display(surface_check_df)


,mask,full_foreground_voxels,surface_only_voxels,removed_interior_voxels,surface_hd95_self_mm,full_mask_hd95_self_mm
0,3x3x3 filled cube,27,26,1,0.0,0.0


In [4]:
if not DATASET_DIR.exists():
    raise FileNotFoundError(f"Dataset directory not found: {DATASET_DIR}")

patient_dirs = sorted(p for p in DATASET_DIR.iterdir() if p.is_dir() and p.name.startswith("TAVI_"))
results = []
skipped_masks = []

for patient_dir in patient_dirs:
    result, skipped = compare_masks(patient_dir)
    if skipped is not None:
        skipped_masks.append(skipped)
        continue
    results.append(result)

metrics_columns = [
    "patient_id",
    "dice",
    "iou",
    "hd95_surface_mm",
    "hd95_full_mask_mm",
    "totalsegmentator_volume_ml",
    "ground_truth_volume_ml",
    "volume_difference_ml",
    "volume_ratio_totalsegmentator_to_gt",
    "totalsegmentator_voxels",
    "ground_truth_voxels",
    "totalsegmentator_surface_voxels",
    "ground_truth_surface_voxels",
    "intersection_voxels",
    "union_voxels",
    "voxel_spacing_mm",
    "totalsegmentator_mask",
    "ground_truth_mask",
]
skipped_columns = ["patient_id", "reason", "path", "error", "pred_shape", "gt_shape"]

metrics_df = pd.DataFrame(results, columns=metrics_columns)
if not metrics_df.empty:
    metrics_df = metrics_df.sort_values("patient_id").reset_index(drop=True)

skipped_df = pd.DataFrame(skipped_masks, columns=skipped_columns)
if not skipped_df.empty:
    skipped_df = skipped_df.sort_values("patient_id").reset_index(drop=True)

metrics_df.to_csv(EVALUATION_OUTPUT_CSV, index=False)
skipped_df.to_csv(SKIPPED_OUTPUT_CSV, index=False)

print(f"Compared masks: {len(metrics_df)}")
print(f"Skipped masks: {len(skipped_df)}")
print(f"Saved metrics to: {EVALUATION_OUTPUT_CSV}")
print(f"Saved skipped-case log to: {SKIPPED_OUTPUT_CSV}")

if not metrics_df.empty:
    summary_cols = [
        "dice",
        "iou",
        "hd95_surface_mm",
        "hd95_full_mask_mm",
        "totalsegmentator_volume_ml",
        "ground_truth_volume_ml",
        "volume_difference_ml",
        "volume_ratio_totalsegmentator_to_gt",
    ]
    display(metrics_df[summary_cols].describe().T)

display(metrics_df)
if not skipped_df.empty:
    display(skipped_df)


Compared masks: 229
Skipped masks: 0
Saved metrics to: /Users/ricca/Desktop/Health_Informatics_Internship_2026/notebooks/total_segmentator_heart_myocardium_metrics.csv
Saved skipped-case log to: /Users/ricca/Desktop/Health_Informatics_Internship_2026/notebooks/total_segmentator_heart_myocardium_skipped.csv


,count,mean,std,min,25%,50%,75%,max
dice,229.0,0.785581,0.049224,0.645082,0.753142,0.790662,0.824407,0.885066
iou,229.0,0.649548,0.066233,0.476104,0.604032,0.653798,0.701269,0.793827
hd95_surface_mm,229.0,4.380738,1.745880,2.745791,3.350615,4.077177,4.971160,24.065100
hd95_full_mask_mm,229.0,3.163152,1.063828,1.111738,2.623756,3.000000,3.538296,9.000000
totalsegmentator_volume_ml,229.0,123.097609,38.511022,59.227677,96.580687,117.398154,139.107813,270.035102
ground_truth_volume_ml,229.0,130.919043,37.310475,62.122566,104.987097,123.849063,148.361443,278.831171
volume_difference_ml,229.0,-7.821434,15.775664,-52.986800,-17.483914,-7.615818,-0.253663,44.922224
volume_ratio_totalsegmentator_to_gt,229.0,0.941763,0.111884,0.701076,0.857705,0.926215,0.997701,1.348695


,patient_id,dice,iou,hd95_surface_mm,hd95_full_mask_mm,totalsegmentator_volume_ml,ground_truth_volume_ml,volume_difference_ml,volume_ratio_totalsegmentator_to_gt,totalsegmentator_voxels,ground_truth_voxels,totalsegmentator_surface_voxels,ground_truth_surface_voxels,intersection_voxels,union_voxels,voxel_spacing_mm,totalsegmentator_mask,ground_truth_mask
0,TAVI_002,0.778047,0.636724,4.365867,3.204078,127.375158,133.747334,-6.372177,0.952357,206789,217134,80219,78232,164916,259007,0.453125x0.453125x3,/Users/ricca/Desktop/Health_Informatics_Intern...,/Users/ricca/Desktop/Health_Informatics_Intern...
1,TAVI_003,0.795087,0.659871,3.888919,3.041056,107.878700,114.572894,-6.694195,0.941573,287738,305593,109510,112879,235875,357456,0.353516x0.353516x3,/Users/ricca/Desktop/Health_Informatics_Intern...,/Users/ricca/Desktop/Health_Informatics_Intern...
2,TAVI_004,0.719466,0.561848,6.751886,5.392771,128.285932,158.297586,-30.011654,0.810410,272023,335661,102432,86428,218604,389080,0.396484x0.396484x3,/Users/ricca/Desktop/Health_Informatics_Intern...,/Users/ricca/Desktop/Health_Informatics_Intern...
3,TAVI_006,0.856186,0.748535,3.000000,1.650391,109.575425,107.856496,1.718929,1.015937,335242,329983,127112,124600,284778,380447,0.330078x0.330078x3,/Users/ricca/Desktop/Health_Informatics_Intern...,/Users/ricca/Desktop/Health_Informatics_Intern...
4,TAVI_007,0.823461,0.699901,3.000000,2.209754,113.172091,100.025178,13.146913,1.131436,285846,252640,107440,111509,221711,316775,0.363281x0.363281x3,/Users/ricca/Desktop/Health_Informatics_Intern...,/Users/ricca/Desktop/Health_Informatics_Intern...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
224,TAVI_372,0.858268,0.751724,3.069419,1.892441,171.883768,173.148400,-1.264632,0.992696,271968,273969,92987,95017,234280,311657,0.458984x0.458984x3,/Users/ricca/Desktop/Health_Informatics_Intern...,/Users/ricca/Desktop/Health_Informatics_Intern...
225,TAVI_373,0.711692,0.552424,6.000000,4.367373,117.398154,156.540738,-39.142584,0.749953,376801,502433,153235,164039,312872,566362,0.322266x0.322266x3,/Users/ricca/Desktop/Health_Informatics_Intern...,/Users/ricca/Desktop/Health_Informatics_Intern...
226,TAVI_374,0.790499,0.653575,4.488218,3.014421,169.333117,180.600321,-11.267203,0.937612,329222,351128,120714,112312,268908,411442,0.414062x0.414062x3,/Users/ricca/Desktop/Health_Informatics_Intern...,/Users/ricca/Desktop/Health_Informatics_Intern...
227,TAVI_376,0.814257,0.686706,3.375145,2.067189,121.924930,134.011353,-12.086423,0.909810,527842,580167,132713,138931,451102,656907,0.339844x0.339844x2,/Users/ricca/Desktop/Health_Informatics_Intern...,/Users/ricca/Desktop/Health_Informatics_Intern...


## Save TotalSegmentator comparison figures

This section mirrors the display style from `leftout_ensemble_inference_visualization.ipynb`: it keeps CT_LATE intensities unchanged, shows an original CT-only view, and uses unlocked aspect for coronal/sagittal panels so they are not vertically squeezed. Green is ground truth only, red is TotalSegmentator only, and yellow is overlap.


In [5]:
import os
from tempfile import gettempdir

MATPLOTLIB_CACHE_DIR = Path(gettempdir()) / "matplotlib-cache"
MATPLOTLIB_CACHE_DIR.mkdir(parents=True, exist_ok=True)
os.environ.setdefault("MPLCONFIGDIR", str(MATPLOTLIB_CACHE_DIR))
os.environ.setdefault("XDG_CACHE_HOME", str(MATPLOTLIB_CACHE_DIR))

import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

VISUALIZATION_OUTPUT_DIR = NOTEBOOK_DIR / "output" / "totalsegmentator_heart_myocardium_visualizations"
VISUALIZATION_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

SLICE_PERCENTAGES = [13, 16, 19, 22, 25, 27, 29, 31, 34, 37, 40, 43, 47, 50, 53, 56, 59, 62, 65, 68, 71, 74, 77, 80]
DISPLAY_FIRST_N_VISUALIZATIONS = 2


def load_ct_volume(ct_path):
    image = nib.load(str(ct_path))
    return np.asanyarray(image.dataobj).astype(np.float32), image.affine


def z_index_for_percent(depth, percent):
    return int(np.clip(round((float(percent) / 100.0) * (depth - 1)), 0, depth - 1))


def _display_ct_slice(ct_slice):
    return np.asarray(ct_slice, dtype=np.float32).T


def _display_mask_slice(mask_slice, display_shape):
    mask_display = np.asarray(mask_slice).T
    if mask_display.shape != tuple(display_shape):
        raise ValueError(
            f"Mask slice shape {mask_display.shape} does not match CT display shape {tuple(display_shape)}"
        )
    return mask_display


def _lock_image_axes(ax, display_shape, aspect="equal"):
    height, width = tuple(display_shape)
    ax.set_xlim(-0.5, width - 0.5)
    ax.set_ylim(-0.5, height - 0.5)
    ax.set_aspect(aspect, adjustable="box")
    ax.axis("off")


def show_ct_only(ax, ct_slice, title, aspect="equal"):
    ct_display = _display_ct_slice(ct_slice)
    ax.imshow(
        ct_display,
        cmap="gray",
        origin="lower",
        aspect=aspect,
        interpolation="nearest",
    )
    ax.set_title(title, fontsize=8)
    _lock_image_axes(ax, ct_display.shape, aspect=aspect)
    return ct_display.shape


def overlay_gt(ax, ct_slice, gt_slice, title, aspect="equal"):
    display_shape = show_ct_only(ax, ct_slice, title, aspect=aspect)
    gt_display = _display_mask_slice(gt_slice, display_shape)
    gt_rgba = np.zeros((*gt_display.shape, 4), dtype=np.float32)
    gt_rgba[gt_display > 0] = [0.0, 1.0, 0.0, 0.55]
    ax.imshow(
        gt_rgba,
        origin="lower",
        extent=(-0.5, display_shape[1] - 0.5, -0.5, display_shape[0] - 0.5),
        aspect=aspect,
        interpolation="nearest",
    )
    _lock_image_axes(ax, display_shape, aspect=aspect)


def overlay_gt_pred(ax, ct_slice, gt_slice, pred_slice, title, aspect="equal"):
    display_shape = show_ct_only(ax, ct_slice, title, aspect=aspect)
    gt_display = _display_mask_slice(gt_slice, display_shape)
    pred_display = _display_mask_slice(pred_slice, display_shape)
    gt_only = (gt_display > 0) & ~(pred_display > 0)
    pred_only = (pred_display > 0) & ~(gt_display > 0)
    overlap = (gt_display > 0) & (pred_display > 0)

    rgba = np.zeros((*gt_display.shape, 4), dtype=np.float32)
    rgba[gt_only] = [0.0, 1.0, 0.0, 0.50]
    rgba[pred_only] = [1.0, 0.1, 0.1, 0.50]
    rgba[overlap] = [1.0, 1.0, 0.0, 0.65]
    ax.imshow(
        rgba,
        origin="lower",
        extent=(-0.5, display_shape[1] - 0.5, -0.5, display_shape[0] - 0.5),
        aspect=aspect,
        interpolation="nearest",
    )
    _lock_image_axes(ax, display_shape, aspect=aspect)


def add_overlay_legend(fig):
    patches = [
        mpatches.Patch(color="green", alpha=0.6, label="Ground truth only"),
        mpatches.Patch(color="red", alpha=0.6, label="TotalSegmentator only"),
        mpatches.Patch(color="yellow", alpha=0.8, label="Overlap"),
    ]
    fig.legend(handles=patches, loc="lower center", ncol=3, fontsize=9, bbox_to_anchor=(0.5, 0.005))


def save_totalsegmentator_slice_percentage_figure(patient_id, ct_vol, gt_mask, pred_mask, metrics, patient_dir, show=False):
    rows = len(SLICE_PERCENTAGES)
    fig, axes = plt.subplots(rows, 3, figsize=(12.6, 2.25 * rows), squeeze=False)
    fig.suptitle(f"{patient_id} | TotalSegmentator myocardium | requested axial z-percent slices", fontsize=13, fontweight="bold")

    for row, percent in enumerate(SLICE_PERCENTAGES):
        z = z_index_for_percent(ct_vol.shape[2], percent)
        show_ct_only(axes[row, 0], ct_vol[:, :, z], f"Original CT_LATE | {percent}% (z={z})")
        overlay_gt(axes[row, 1], ct_vol[:, :, z], gt_mask[:, :, z], f"Ground truth | {percent}% (z={z})")
        title = (
            f"TotalSegmentator | {percent}% (z={z})\n"
            f"Dice {metrics['dice']:.3f} | surface HD95 {metrics['hd95_surface_mm']:.1f} mm"
        )
        overlay_gt_pred(axes[row, 2], ct_vol[:, :, z], gt_mask[:, :, z], pred_mask[:, :, z], title)

    add_overlay_legend(fig)
    plt.tight_layout(rect=(0, 0.025, 1, 0.985))
    figure_path = patient_dir / "slices_z_percentages_totalsegmentator.png"
    fig.savefig(figure_path, dpi=150, bbox_inches="tight")
    if show:
        plt.show()
    else:
        plt.close(fig)
    return figure_path


def centroid_slice(mask):
    coords = np.argwhere(mask > 0)
    if len(coords) == 0:
        return tuple((np.array(mask.shape) // 2).astype(int))
    return tuple(coords.mean(axis=0).astype(int))


def save_totalsegmentator_three_plane_figure(patient_id, ct_vol, gt_mask, pred_mask, metrics, patient_dir, show=False):
    cx, cy, cz = centroid_slice(gt_mask)
    fig, axes = plt.subplots(2, 3, figsize=(13, 8.6), squeeze=False)
    fig.suptitle(f"{patient_id} | TotalSegmentator centroid three-plane comparison", fontsize=13, fontweight="bold")

    planes = [
        (f"Axial z={cz}", ct_vol[:, :, cz], gt_mask[:, :, cz], pred_mask[:, :, cz]),
        (f"Coronal y={cy}", ct_vol[:, cy, :], gt_mask[:, cy, :], pred_mask[:, cy, :]),
        (f"Sagittal x={cx}", ct_vol[cx, :, :], gt_mask[cx, :, :], pred_mask[cx, :, :]),
    ]
    for col, (plane_name, ct_slice, gt_slice, pred_slice) in enumerate(planes):
        plane_aspect = "equal" if col == 0 else "auto"
        show_ct_only(axes[0, col], ct_slice, f"Original CT_LATE | {plane_name}", aspect=plane_aspect)
        title = f"TotalSegmentator | {plane_name}\nDice {metrics['dice']:.3f} | full HD95 {metrics['hd95_full_mask_mm']:.1f} mm"
        overlay_gt_pred(axes[1, col], ct_slice, gt_slice, pred_slice, title, aspect=plane_aspect)

    add_overlay_legend(fig)
    plt.tight_layout(rect=(0, 0.04, 1, 0.94))
    figure_path = patient_dir / "three_plane_totalsegmentator.png"
    fig.savefig(figure_path, dpi=150, bbox_inches="tight")
    if show:
        plt.show()
    else:
        plt.close(fig)
    return figure_path


def load_patient_visualization_inputs(patient_dir):
    ct_path = patient_dir / "CT_LATE.nii.gz"
    pred_path = patient_dir / TOTAL_SEGMENTATOR_MASK_RELATIVE_PATH
    gt_path = patient_dir / GROUND_TRUTH_MASK_NAME

    ct_vol, ct_affine = load_ct_volume(ct_path)
    pred_mask, pred_affine, _ = load_binary_mask(pred_path)
    gt_mask, gt_affine, _ = load_binary_mask(gt_path)

    if ct_vol.shape != pred_mask.shape or ct_vol.shape != gt_mask.shape:
        raise ValueError(
            f"Shape mismatch for {patient_dir.name}: "
            f"CT {ct_vol.shape}, TotalSegmentator {pred_mask.shape}, GT {gt_mask.shape}"
        )
    if not np.allclose(ct_affine, pred_affine, rtol=1e-3, atol=1e-3):
        raise ValueError(f"CT and TotalSegmentator affine mismatch for {patient_dir.name}")
    if not np.allclose(ct_affine, gt_affine, rtol=1e-3, atol=1e-3):
        raise ValueError(f"CT and ground-truth affine mismatch for {patient_dir.name}")

    return ct_vol, gt_mask.astype(np.uint8), pred_mask.astype(np.uint8)


In [6]:
if "metrics_df" not in globals():
    raise RuntimeError("Run the metrics comparison cell before generating figures.")

visualization_rows = []
visualization_skipped = []

for row_index, metric_row in metrics_df.iterrows():
    patient_id = metric_row["patient_id"]
    patient_dir = DATASET_DIR / patient_id
    patient_output_dir = VISUALIZATION_OUTPUT_DIR / patient_id
    patient_output_dir.mkdir(parents=True, exist_ok=True)

    try:
        ct_vol, gt_mask, pred_mask = load_patient_visualization_inputs(patient_dir)
        show_figures = row_index < DISPLAY_FIRST_N_VISUALIZATIONS
        slice_figure = save_totalsegmentator_slice_percentage_figure(
            patient_id, ct_vol, gt_mask, pred_mask, metric_row, patient_output_dir, show=show_figures
        )
        three_plane_figure = save_totalsegmentator_three_plane_figure(
            patient_id, ct_vol, gt_mask, pred_mask, metric_row, patient_output_dir, show=show_figures
        )
    except (ImageFileError, OSError, ValueError) as exc:
        visualization_skipped.append({"patient_id": patient_id, "reason": str(exc)})
        continue

    visualization_rows.append(
        {
            "patient_id": patient_id,
            "patient_dir": str(patient_output_dir),
            "slice_figure": str(slice_figure),
            "three_plane_figure": str(three_plane_figure),
            "ct_volume": str(patient_dir / "CT_LATE.nii.gz"),
            "ground_truth_mask": str(patient_dir / GROUND_TRUTH_MASK_NAME),
            "totalsegmentator_mask": str(patient_dir / TOTAL_SEGMENTATOR_MASK_RELATIVE_PATH),
        }
    )

visualization_df = pd.DataFrame(visualization_rows)
visualization_index_csv = VISUALIZATION_OUTPUT_DIR / "totalsegmentator_visualization_index.csv"
visualization_df.to_csv(visualization_index_csv, index=False)

visualization_skipped_df = pd.DataFrame(visualization_skipped)
visualization_skipped_csv = VISUALIZATION_OUTPUT_DIR / "totalsegmentator_visualization_skipped.csv"
visualization_skipped_df.to_csv(visualization_skipped_csv, index=False)

print(f"Saved visualizations for {len(visualization_df)} patients to: {VISUALIZATION_OUTPUT_DIR}")
print(f"Saved visualization index to: {visualization_index_csv}")
if not visualization_skipped_df.empty:
    print(f"Skipped {len(visualization_skipped_df)} visualization cases. Log: {visualization_skipped_csv}")

display(visualization_df)
if not visualization_skipped_df.empty:
    display(visualization_skipped_df)


/var/folders/zk/kwxz8r6s39d68p2qk18gzn840000gn/T/ipykernel_1451/222514978.py:129: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
/var/folders/zk/kwxz8r6s39d68p2qk18gzn840000gn/T/ipykernel_1451/222514978.py:163: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


Saved visualizations for 229 patients to: /Users/ricca/Desktop/Health_Informatics_Internship_2026/notebooks/output/totalsegmentator_heart_myocardium_visualizations
Saved visualization index to: /Users/ricca/Desktop/Health_Informatics_Internship_2026/notebooks/output/totalsegmentator_heart_myocardium_visualizations/totalsegmentator_visualization_index.csv


,patient_id,patient_dir,slice_figure,three_plane_figure,ct_volume,ground_truth_mask,totalsegmentator_mask
0,TAVI_002,/Users/ricca/Desktop/Health_Informatics_Intern...,/Users/ricca/Desktop/Health_Informatics_Intern...,/Users/ricca/Desktop/Health_Informatics_Intern...,/Users/ricca/Desktop/Health_Informatics_Intern...,/Users/ricca/Desktop/Health_Informatics_Intern...,/Users/ricca/Desktop/Health_Informatics_Intern...
1,TAVI_003,/Users/ricca/Desktop/Health_Informatics_Intern...,/Users/ricca/Desktop/Health_Informatics_Intern...,/Users/ricca/Desktop/Health_Informatics_Intern...,/Users/ricca/Desktop/Health_Informatics_Intern...,/Users/ricca/Desktop/Health_Informatics_Intern...,/Users/ricca/Desktop/Health_Informatics_Intern...
2,TAVI_004,/Users/ricca/Desktop/Health_Informatics_Intern...,/Users/ricca/Desktop/Health_Informatics_Intern...,/Users/ricca/Desktop/Health_Informatics_Intern...,/Users/ricca/Desktop/Health_Informatics_Intern...,/Users/ricca/Desktop/Health_Informatics_Intern...,/Users/ricca/Desktop/Health_Informatics_Intern...
3,TAVI_006,/Users/ricca/Desktop/Health_Informatics_Intern...,/Users/ricca/Desktop/Health_Informatics_Intern...,/Users/ricca/Desktop/Health_Informatics_Intern...,/Users/ricca/Desktop/Health_Informatics_Intern...,/Users/ricca/Desktop/Health_Informatics_Intern...,/Users/ricca/Desktop/Health_Informatics_Intern...
4,TAVI_007,/Users/ricca/Desktop/Health_Informatics_Intern...,/Users/ricca/Desktop/Health_Informatics_Intern...,/Users/ricca/Desktop/Health_Informatics_Intern...,/Users/ricca/Desktop/Health_Informatics_Intern...,/Users/ricca/Desktop/Health_Informatics_Intern...,/Users/ricca/Desktop/Health_Informatics_Intern...
...,...,...,...,...,...,...,...
224,TAVI_372,/Users/ricca/Desktop/Health_Informatics_Intern...,/Users/ricca/Desktop/Health_Informatics_Intern...,/Users/ricca/Desktop/Health_Informatics_Intern...,/Users/ricca/Desktop/Health_Informatics_Intern...,/Users/ricca/Desktop/Health_Informatics_Intern...,/Users/ricca/Desktop/Health_Informatics_Intern...
225,TAVI_373,/Users/ricca/Desktop/Health_Informatics_Intern...,/Users/ricca/Desktop/Health_Informatics_Intern...,/Users/ricca/Desktop/Health_Informatics_Intern...,/Users/ricca/Desktop/Health_Informatics_Intern...,/Users/ricca/Desktop/Health_Informatics_Intern...,/Users/ricca/Desktop/Health_Informatics_Intern...
226,TAVI_374,/Users/ricca/Desktop/Health_Informatics_Intern...,/Users/ricca/Desktop/Health_Informatics_Intern...,/Users/ricca/Desktop/Health_Informatics_Intern...,/Users/ricca/Desktop/Health_Informatics_Intern...,/Users/ricca/Desktop/Health_Informatics_Intern...,/Users/ricca/Desktop/Health_Informatics_Intern...
227,TAVI_376,/Users/ricca/Desktop/Health_Informatics_Intern...,/Users/ricca/Desktop/Health_Informatics_Intern...,/Users/ricca/Desktop/Health_Informatics_Intern...,/Users/ricca/Desktop/Health_Informatics_Intern...,/Users/ricca/Desktop/Health_Informatics_Intern...,/Users/ricca/Desktop/Health_Informatics_Intern...
